# 03 - Gaussian Naive Bayes: Pelatihan Model

## 5. Metode Komputasi Soft Computing yang Digunakan

**Naive Bayes (Gaussian Naive Bayes).** Target klasifikasi adalah
`DiagnosisType` (Normal / Tipe 2 / Gestasional), hasil penggabungan
Pima + GDM pada `02_Preprocessing`, berbasis diagnosis klinis asli
kedua studi. Tahapan:

1. Pembagian data latih/uji, lalu **pengisian nilai hilang** memakai
   median data latih -- tanpa melihat label maupun data uji.
2. Penghitungan **probabilitas prior** tiap kelas dari data latih.
3. Estimasi **distribusi normal** (mean, variansi) tiap atribut per
   kelas.
4. Penerapan **Teorema Bayes** untuk probabilitas posterior
   `P(kelas | atribut)` pada data uji.
5. Pemilihan kelas dengan probabilitas posterior tertinggi.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Harus sama dengan 04_Evaluation, yang membentuk ulang split ini.
RANDOM_STATE = 42
TEST_SIZE = 0.2

df_gabungan = pd.read_csv(PROCESSED_DIR / "merged_diagnosis.csv")
df_model = df_gabungan.drop(columns=["SumberData"])

print(f"Dataset model: {df_model.shape}")
print("Nilai hilang per kolom (belum diisi):", df_model.isna().sum().to_dict())
print(f"random_state={RANDOM_STATE}, test_size={TEST_SIZE}")

Dataset model: (4293, 6)
Nilai hilang per kolom (belum diisi): {'Age': 0, 'Pregnancies': 0, 'BMI': 1092, 'BloodPressure': 35, 'Glucose': 518, 'DiagnosisType': 0}
random_state=42, test_size=0.2


## Membagi Data, Mengisi Nilai Hilang, dan Melatih Model

Data dibagi lebih dulu (80:20, stratifikasi pada label agar proporsi
kelas terjaga), lalu median tiap fitur dihitung **hanya dari data
latih** dan dipakai mengisi nilai hilang data latih maupun data uji,
baru kemudian model dilatih. Urutan ini mencegah label maupun data uji
ikut menentukan nilai pengisi.

In [2]:
X = df_model.drop(columns=["DiagnosisType"])
y = df_model["DiagnosisType"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

median_latih = {kolom: float(X_train[kolom].median()) for kolom in X_train.columns}
X_train = X_train.fillna(value=median_latih)
X_test = X_test.fillna(value=median_latih)

model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Dilatih pada", X_train.shape[0], "sampel latih,", X_test.shape[0], "sampel uji")
print("Median data latih untuk imputasi:", median_latih)
print("Sisa nilai hilang (latih, uji):", int(X_train.isna().sum().sum()), int(X_test.isna().sum().sum()))
print("Kelas yang dipelajari model.classes_:", model.classes_.tolist())

Dilatih pada 3434 sampel latih, 859 sampel uji
Median data latih untuk imputasi: {'Age': 32.0, 'Pregnancies': 2.0, 'BMI': 28.4, 'BloodPressure': 80.0, 'Glucose': 154.0}
Sisa nilai hilang (latih, uji): 0 0
Kelas yang dipelajari model.classes_: ['Gestasional', 'Normal', 'Tipe 2']


## 5.1 Probabilitas Prior Tiap Kelas


In [3]:
prior_model = pd.Series(model.class_prior_, index=model.classes_, name="prior_dari_model")
prior_manual = y_train.value_counts(normalize=True).sort_index()
prior_manual.name = "prior_manual"

tabel_prior = pd.concat([prior_model, prior_manual], axis=1)
tabel_prior["cocok"] = np.isclose(tabel_prior["prior_dari_model"], tabel_prior["prior_manual"])
tabel_prior

,prior_dari_model,prior_manual,cocok
Gestasional,0.319744,0.319744,True
Normal,0.617938,0.617938,True
Tipe 2,0.062318,0.062318,True


Prior `Tipe 2` jauh lebih kecil (~6%) dibanding `Normal` (~62%) dan
`Gestasional` (~32%) -- konsekuensi langsung dari Pima (768 pasien)
jauh lebih kecil daripada GDM (3.525 pasien). Ketidakseimbangan ini
dibahas lebih lanjut pada `04_Evaluation`.


## 5.2 Estimasi Distribusi Normal (Mean dan Variansi) per Atribut per Kelas


In [4]:
fitur = list(X_train.columns)
distribusi = pd.DataFrame(
    {f"mean_{kelas}": model.theta_[i] for i, kelas in enumerate(model.classes_)},
    index=fitur,
)
for i, kelas in enumerate(model.classes_):
    distribusi[f"var_{kelas}"] = model.var_[i]
distribusi.round(2)

,mean_Gestasional,mean_Normal,mean_Tipe 2,var_Gestasional,var_Normal,var_Tipe 2
Age,36.09,30.52,36.84,35.62,47.11,123.14
Pregnancies,2.30,2.19,4.83,0.78,2.50,13.55
BMI,31.11,26.88,35.52,19.69,22.97,40.23
BloodPressure,89.96,75.36,75.88,116.54,85.95,144.71
Glucose,207.26,137.30,143.12,2158.13,557.34,883.68


`BMI` menunjukkan mean yang jauh berbeda antara kelompok asal-Pima
(`Normal`, `Tipe 2`) dan `Gestasional` (asal-GDM) -- konsisten dengan
temuan `01_EDA`/`02_Preprocessing` bahwa `BMI` sebagian membawa sinyal
sumber data, bukan murni tipe diabetes.


## 5.3 Penerapan Teorema Bayes: Menghitung Posterior Secara Manual

$$P(y \mid x_1, \dots, x_5) \propto P(y) \prod_{i=1}^{5} P(x_i \mid y)$$


In [5]:
def log_pdf_normal(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)


def posterior_manual(sample, model):
    log_prior = np.log(model.class_prior_)
    log_likelihood = np.array([
        log_pdf_normal(sample.values, model.theta_[k], model.var_[k]).sum()
        for k in range(len(model.classes_))
    ])
    joint = log_prior + log_likelihood
    joint_stabil = joint - joint.max()
    probs = np.exp(joint_stabil)
    probs = probs / probs.sum()
    return probs


sampel = X_test.iloc[0]
label_sebenarnya = y_test.iloc[0]

posterior_manual_hasil = posterior_manual(sampel, model)
posterior_sklearn = model.predict_proba(sampel.to_frame().T)[0]

print("Sampel data uji (indeks 0):")
print(sampel.to_string())
print()
print(f"Label sebenarnya (DiagnosisType): {label_sebenarnya}")
print()
print(f"{'Kelas':<14}{'Posterior manual':<20}{'Posterior sklearn':<20}")
for kelas, p_manual, p_sklearn in zip(model.classes_, posterior_manual_hasil, posterior_sklearn):
    print(f"{kelas:<14}{p_manual:<20.6f}{p_sklearn:<20.6f}")

assert np.allclose(posterior_manual_hasil, posterior_sklearn, atol=1e-6), "Posterior manual tidak cocok dengan sklearn!"
print()
print("Posterior manual dan hasil predict_proba() sklearn identik (selisih < 1e-6).")

Sampel data uji (indeks 0):
Age               34.0
Pregnancies        2.0
BMI               27.0
BloodPressure     86.0
Glucose          171.0

Label sebenarnya (DiagnosisType): Gestasional

Kelas         Posterior manual    Posterior sklearn   
Gestasional   0.548224            0.548224            
Normal        0.447290            0.447290            
Tipe 2        0.004486            0.004486            

Posterior manual dan hasil predict_proba() sklearn identik (selisih < 1e-6).


## 5.4 Keputusan Kelas


In [6]:
prediksi_manual = np.array([
    model.classes_[np.argmax(posterior_manual(X_test.iloc[i], model))]
    for i in range(len(X_test))
])
prediksi_sklearn = y_pred
cocok = (prediksi_manual == prediksi_sklearn).all()
print(f"Prediksi manual identik dengan model.predict() untuk seluruh {len(prediksi_sklearn)} data uji: {cocok}")

Prediksi manual identik dengan model.predict() untuk seluruh 859 data uji: True


## Menyimpan Model dan Artefak Deployment


In [7]:
import json

joblib.dump(model, MODELS_DIR / "gnb_diagnosis.joblib")

with open(MODELS_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(fitur, f, indent=2)

with open(MODELS_DIR / "imputation_medians.json", "w", encoding="utf-8") as f:
    json.dump(median_latih, f, indent=2)

print("Tersimpan:")
print(" ", MODELS_DIR / "gnb_diagnosis.joblib")
print(" ", MODELS_DIR / "feature_columns.json", "->", fitur)
print(" ", MODELS_DIR / "imputation_medians.json", "->", median_latih)

Tersimpan:
  D:\diabetes-risk-gnb\models\gnb_diagnosis.joblib
  D:\diabetes-risk-gnb\models\feature_columns.json -> ['Age', 'Pregnancies', 'BMI', 'BloodPressure', 'Glucose']
  D:\diabetes-risk-gnb\models\imputation_medians.json -> {'Age': 32.0, 'Pregnancies': 2.0, 'BMI': 28.4, 'BloodPressure': 80.0, 'Glucose': 154.0}


**Lanjut ke `04_Evaluation.ipynb`** untuk evaluasi kuantitatif dan
pembuktian dampak perancu sumber data.
